In [3]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    accuracy_score,
    f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.over_sampling import RandomOverSampler


def run_logistic_model(x_train_path, x_test_path, y_train_path, y_test_path, upsample=False):
    # 1. read data
    X_train = pd.read_csv(x_train_path)
    X_test = pd.read_csv(x_test_path)
    y_train = pd.read_csv(y_train_path)["y"]
    y_test = pd.read_csv(y_test_path)["y"]

    # 2. define model and CV setting
    log_model = LogisticRegression(max_iter=5000, random_state=42)

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # 3. fivefold cross-validation on the training set
    if upsample:
        cv_scores = {
            "AUC": [],
            "Precision": [],
            "Recall": [],
            "Accuracy": [],
            "F1": []
        }

        for train_idx, val_idx in cv.split(X_train, y_train):
            X_train_fold = X_train.iloc[train_idx]
            y_train_fold = y_train.iloc[train_idx]

            X_val_fold = X_train.iloc[val_idx]
            y_val_fold = y_train.iloc[val_idx]

            # apply random over-sampling only to the training fold
            upsampler = RandomOverSampler(random_state=42)
            X_train_fold_up, y_train_fold_up = upsampler.fit_resample(
                X_train_fold,
                y_train_fold
            )

            log_model.fit(X_train_fold_up, y_train_fold_up)

            y_val_pred = log_model.predict(X_val_fold)
            y_val_prob = log_model.predict_proba(X_val_fold)[:, 1]

            cv_scores["AUC"].append(roc_auc_score(y_val_fold, y_val_prob))
            cv_scores["Precision"].append(
                precision_score(y_val_fold, y_val_pred, zero_division=0)
            )
            cv_scores["Recall"].append(
                recall_score(y_val_fold, y_val_pred, zero_division=0)
            )
            cv_scores["Accuracy"].append(
                accuracy_score(y_val_fold, y_val_pred)
            )
            cv_scores["F1"].append(
                f1_score(y_val_fold, y_val_pred, zero_division=0)
            )

        cv_metrics_df = pd.DataFrame({
            "Metric": ["AUC", "Precision", "Recall", "Accuracy", "F1"],
            "Mean CV Score": [
                np.mean(cv_scores["AUC"]),
                np.mean(cv_scores["Precision"]),
                np.mean(cv_scores["Recall"]),
                np.mean(cv_scores["Accuracy"]),
                np.mean(cv_scores["F1"])
            ],
            "CV Std": [
                np.std(cv_scores["AUC"], ddof=1),
                np.std(cv_scores["Precision"], ddof=1),
                np.std(cv_scores["Recall"], ddof=1),
                np.std(cv_scores["Accuracy"], ddof=1),
                np.std(cv_scores["F1"], ddof=1)
            ]
        }).round(3)

    else:
        scoring = ["roc_auc", "precision", "recall", "accuracy", "f1"]

        cv_results = cross_validate(
            log_model,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring
        )

        cv_metrics_df = pd.DataFrame({
            "Metric": ["AUC", "Precision", "Recall", "Accuracy", "F1"],
            "Mean CV Score": [
                cv_results["test_roc_auc"].mean(),
                cv_results["test_precision"].mean(),
                cv_results["test_recall"].mean(),
                cv_results["test_accuracy"].mean(),
                cv_results["test_f1"].mean()
            ],
            "CV Std": [
                cv_results["test_roc_auc"].std(ddof=1),
                cv_results["test_precision"].std(ddof=1),
                cv_results["test_recall"].std(ddof=1),
                cv_results["test_accuracy"].std(ddof=1),
                cv_results["test_f1"].std(ddof=1)
            ]
        }).round(3)

    # 4. fit final model on the full training set
    if upsample:
        upsampler = RandomOverSampler(random_state=42)
        X_train_final, y_train_final = upsampler.fit_resample(
            X_train,
            y_train
        )
        log_model.fit(X_train_final, y_train_final)
    else:
        log_model.fit(X_train, y_train)

    # 5. predict on independent test set
    y_pred = log_model.predict(X_test)
    y_prob = log_model.predict_proba(X_test)[:, 1]

    # 6. test set metrics
    test_metrics_df = pd.DataFrame({
        "Metric": ["AUC", "Precision", "Recall", "Accuracy", "F1"],
        "Test Score": [
            roc_auc_score(y_test, y_prob),
            precision_score(y_test, y_pred, zero_division=0),
            recall_score(y_test, y_pred, zero_division=0),
            accuracy_score(y_test, y_pred),
            f1_score(y_test, y_pred, zero_division=0)
        ]
    }).round(3)

    return {
        "cv_metrics": cv_metrics_df,
        "test_metrics": test_metrics_df
    }

In [4]:
def make_cv_summary(results_dict):
    # create a summary table for cross-validation results
    rows = []

    for model_name, result in results_dict.items():
        cv_df = result["cv_metrics"]

        row = {"Model": model_name}

        for metric in ["AUC", "Precision", "Recall", "Accuracy", "F1"]:
            mean_value = cv_df.loc[
                cv_df["Metric"] == metric,
                "Mean CV Score"
            ].values[0]

            cv_std = cv_df.loc[
                cv_df["Metric"] == metric,
                "CV Std"
            ].values[0]

            # Standard error across five folds
            cv_se = cv_std / np.sqrt(5)

            # report mean CV score with standard error in parentheses
            row[metric] = f"{mean_value:.3f} ({cv_se:.3f})"

        rows.append(row)

    return pd.DataFrame(rows)


def make_test_summary(results_dict):
    # create a summary table for independent test set results
    rows = []

    for model_name, result in results_dict.items():
        test_df = result["test_metrics"]

        row = {"Model": model_name}

        for metric in ["AUC", "Precision", "Recall", "Accuracy", "F1"]:
            test_value = test_df.loc[
                test_df["Metric"] == metric,
                "Test Score"
            ].values[0]

            row[metric] = round(test_value, 3)

        rows.append(row)

    return pd.DataFrame(rows)

In [5]:
# Run random forest models for Model 1
result_1 = run_logistic_model(
    x_train_path="../data/model_inputs/X_train_model_1.csv",
    x_test_path="../data/model_inputs/X_test_model_1.csv",
    y_train_path="../data/model_inputs/y_train_model_1.csv",
    y_test_path="../data/model_inputs/y_test_model_1.csv",
    upsample=False
)

result_1a = run_logistic_model(
    x_train_path="../data/model_inputs/X_train_model_1a.csv",
    x_test_path="../data/model_inputs/X_test_model_1a.csv",
    y_train_path="../data/model_inputs/y_train_model_1a.csv",
    y_test_path="../data/model_inputs/y_test_model_1a.csv",
    upsample=True
)

result_1b = run_logistic_model(
    x_train_path="../data/model_inputs/X_train_model_1b.csv",
    x_test_path="../data/model_inputs/X_test_model_1b.csv",
    y_train_path="../data/model_inputs/y_train_model_1b.csv",
    y_test_path="../data/model_inputs/y_test_model_1b.csv",
    upsample=True
)

In [6]:
# Run random forest models for Model 2
result_2 = run_logistic_model(
    x_train_path="../data/model_inputs/X_train_model_2.csv",
    x_test_path="../data/model_inputs/X_test_model_2.csv",
    y_train_path="../data/model_inputs/y_train_model_2.csv",
    y_test_path="../data/model_inputs/y_test_model_2.csv",
    upsample=False
)

result_2a = run_logistic_model(
    x_train_path="../data/model_inputs/X_train_model_2a.csv",
    x_test_path="../data/model_inputs/X_test_model_2a.csv",
    y_train_path="../data/model_inputs/y_train_model_2a.csv",
    y_test_path="../data/model_inputs/y_test_model_2a.csv",
    upsample=True
)

result_2b = run_logistic_model(
    x_train_path="../data/model_inputs/X_train_model_2b.csv",
    x_test_path="../data/model_inputs/X_test_model_2b.csv",
    y_train_path="../data/model_inputs/y_train_model_2b.csv",
    y_test_path="../data/model_inputs/y_test_model_2b.csv",
    upsample=True
)

In [7]:
# Store Model 1 results
lr_model1_results = {
    "LR Model 1": result_1,
    "LR Model 1a": result_1a,
    "LR Model 1b": result_1b
}

# Store Model 2 results
lr_model2_results = {
    "LR Model 2": result_2,
    "LR Model 2a": result_2a,
    "LR Model 2b": result_2b
}

In [8]:
# Create summary tables for Model 1
cv_summary_model1_df = make_cv_summary(lr_model1_results)
test_summary_model1_df = make_test_summary(lr_model1_results)

print("\n===== CV Summary Table (lr model 1) =====")
print(cv_summary_model1_df.to_string(index=False))

print("\n===== Test Summary Table (lr model 1) =====")
print(test_summary_model1_df.to_string(index=False))

# Save summary tables for Model 1
cv_summary_model1_df.to_csv("../results/lr_model1_cv_summary.csv", index=False)
test_summary_model1_df.to_csv("../results/lr_model1_test_summary.csv", index=False)


===== CV Summary Table (lr model 1) =====
      Model           AUC     Precision        Recall      Accuracy            F1
 LR Model 1 0.871 (0.001) 0.802 (0.002) 0.821 (0.002) 0.794 (0.001) 0.811 (0.001)
LR Model 1a 0.818 (0.003) 0.480 (0.004) 0.765 (0.009) 0.726 (0.004) 0.590 (0.005)
LR Model 1b 0.836 (0.003) 0.880 (0.004) 0.781 (0.002) 0.770 (0.003) 0.827 (0.002)

===== Test Summary Table (lr model 1) =====
      Model   AUC  Precision  Recall  Accuracy    F1
 LR Model 1 0.867      0.806   0.818     0.795 0.812
LR Model 1a 0.820      0.484   0.758     0.726 0.591
LR Model 1b 0.828      0.877   0.769     0.760 0.819


In [9]:
# Create summary tables for Model 2
cv_summary_model2_df = make_cv_summary(lr_model2_results)
test_summary_model2_df = make_test_summary(lr_model2_results)

print("\n===== CV Summary Table (lr model 2) =====")
print(cv_summary_model2_df.to_string(index=False))

print("\n===== Test Summary Table (lr model 2) =====")
print(test_summary_model2_df.to_string(index=False))

# Save summary tables for Model 2
cv_summary_model2_df.to_csv("../results/lr_model2_cv_summary.csv", index=False)
test_summary_model2_df.to_csv("../results/lr_model2_test_summary.csv", index=False)


===== CV Summary Table (lr model 2) =====
      Model           AUC     Precision        Recall      Accuracy            F1
 LR Model 2 0.778 (0.002) 0.755 (0.002) 0.867 (0.001) 0.730 (0.002) 0.807 (0.001)
LR Model 2a 0.732 (0.003) 0.687 (0.003) 0.692 (0.004) 0.671 (0.003) 0.690 (0.003)
LR Model 2b 0.785 (0.003) 0.864 (0.003) 0.734 (0.004) 0.723 (0.003) 0.794 (0.003)

===== Test Summary Table (lr model 2) =====
      Model   AUC  Precision  Recall  Accuracy    F1
 LR Model 2 0.768      0.748   0.870     0.724 0.804
LR Model 2a 0.732      0.694   0.716     0.680 0.705
LR Model 2b 0.774      0.856   0.727     0.715 0.786
